## 1. Importar Bibliotecas Necessárias

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. Carregar e Explorar os Dados

Utilizaremos um dataset de churn de clientes de telecomunicações.

In [ ]:
# Carregar dataset
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/refs/heads/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

In [ ]:

# Visualizar primeiras linhas
df.head()

In [ ]:
# Informações gerais
df.info()

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Verificar valores faltantes
df.isnull().sum()

In [ ]:
# Distribuição da variável alvo (Churn)
print(df['Churn'].value_counts())
print(f"\nProporção: {df['Churn'].value_counts(normalize=True)}")

## 3. Preparação dos Dados

In [ ]:
# Criar uma cópia do dataframe
df_clean = df.copy()

# Remover coluna de ID do cliente (não é útil para predição)
df_clean = df_clean.drop('customerID', axis=1)

# Converter 'TotalCharges' para numérico (há alguns valores em branco)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# Preencher valores faltantes com a mediana
df_clean['TotalCharges'].fillna(df_clean['TotalCharges'].median(), inplace=True)

# Converter variável alvo (Churn) para binária (0 e 1)
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)

print(f"\nForma do dataset: {df_clean.shape}")
print(f"Variável alvo (Churn): {df_clean['Churn'].value_counts().to_dict()}")
print("Dataset após limpeza:")
df_clean.head()


## 4. Engenharia de Features

In [ ]:
# Identificar variáveis categóricas e numéricas
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols.remove('Churn')  # Remover a variável alvo da lista de numéricas

print(f"Variáveis categóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"Variáveis numéricas ({len(numeric_cols)}): {numeric_cols}")

In [ ]:
# Codificar variáveis categóricas
df_encoded = df_clean.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f"\n{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\n" + "="*80)
print("Dataset após codificação:")
df_encoded.head()

## 5. Dividir Dados em Treino, Validação e Teste

Usaremos a proporção: 70% treino, 10% validação, 20% teste

In [ ]:
# Separar features e target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

qt_samples_test = int(0.20 * len(X))
qt_samples_val = int(0.10 * len(X))

# Primeira divisão: 70% treino + validação, 30% teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, 
    test_size=qt_samples_test, 
    random_state=42, 
    stratify=y
)

# Segunda divisão: dividir os 80% em 70% treino e 10% validação
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=qt_samples_val,
    random_state=42,
    stratify=y_temp
)

print("Divisão dos dados:")
print(f"Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.2f}%)")
print(f"Validação: {X_val.shape[0]} amostras ({X_val.shape[0]/len(X)*100:.2f}%)")
print(f"Teste: {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.2f}%)")

print("\n" + "="*80)
print("Distribuição do target em cada conjunto:")
print(f"\nTreino - Churn=0: {(y_train==0).sum()} {round((y_train==0).sum()/len(y_train)*100,1)}% | Churn=1: {(y_train==1).sum()}")
print(f"Validação - Churn=0: {(y_val==0).sum()} {round((y_val==0).sum()/len(y_val)*100,1)}% | Churn=1: {(y_val==1).sum()}")
print(f"Teste - Churn=0: {(y_test==0).sum()} {round((y_test==0).sum()/len(y_test)*100,1)}% | Churn=1: {(y_test==1).sum()}")

## 6. Escalar Features Numéricas

In [ ]:
# Obs. O XGBoost não requer padronização/normalização dos dados, pois é baseado em árvores de decisão.
# entretanto, para fins didáticos, vamos aplicar a padronização.

In [ ]:
# Aplicar StandardScaler nas features numéricas
scaler = StandardScaler()

# Escalar treino
X_train_scaled = X_train.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# Escalar validação
X_val_scaled = X_val.copy()
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])

# Escalar teste
X_test_scaled = X_test.copy()
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Features escaladas!")
print(f"Treino escalado:")
X_train_scaled.head()

## 7. Treinar Modelo XGBoost

In [ ]:
# Criar e treinar modelo XGBoost
modelo_xgb = xgb.XGBClassifier(
    n_estimators=100,           # Número de árvores
    max_depth=6,                # Profundidade máxima da árvore
    learning_rate=0.1,          # Taxa de aprendizado
    subsample=0.8,              # Fração de amostras para treinar cada árvore
    colsample_bytree=0.8,       # Fração de features para cada árvore
    objective='binary:logistic', # Classificação binária
    random_state=42,
    verbosity=1
)

# Treinar no conjunto de treino
print("Treinando modelo XGBoost...")
modelo_xgb.fit(
    X_train_scaled, y_train,
    eval_set=[(X_train_scaled, y_train), (X_val_scaled, y_val)],
    verbose=True
)

print("Modelo treinado com sucesso!")

## 8. Fazer Predições

In [ ]:
y_test_pred = modelo_xgb.predict(X_test_scaled)

# Probabilidades
y_test_pred_proba = modelo_xgb.predict_proba(X_test_scaled)[:, 1]

print("Predições realizadas!")
print(f"\nExemplos de predições (teste):")
print(f"Primeiros 10 valores reais: {y_test.values[:10]}")
print(f"Primeiros 10 predições: {y_test_pred[:10]}")
print(f"Primeiros 10 probabilidades: {y_test_pred_proba[:10].round(4)}")

## 9. Avaliar Modelo - Métricas de Desempenho

In [ ]:
y_train_pred = modelo_xgb.predict(X_train_scaled)
y_val_pred = modelo_xgb.predict(X_val_scaled)

y_train_pred_proba = modelo_xgb.predict_proba(X_train_scaled)[:, 1]
y_val_pred_proba = modelo_xgb.predict_proba(X_val_scaled)[:, 1]

In [ ]:
# Função para calcular métricas
def avaliar_modelo(y_true, y_pred, y_pred_proba, conjunto_nome):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    
    print(f"\n{'='*60}")
    print(f"MÉTRICAS - {conjunto_nome}")
    print(f"{'='*60}")
    print(f"Acurácia:     {acc:.4f} ({acc*100:.2f}%)")
    print(f"Precisão:     {prec:.4f} ({prec*100:.2f}%)")
    print(f"Recall:       {rec:.4f} ({rec*100:.2f}%)")
    print(f"F1-Score:     {f1:.4f}")
    print(f"ROC-AUC:      {roc_auc:.4f}")
    
    return {'Acurácia': acc, 'Precisão': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': roc_auc}

# Avaliar em todos os conjuntos
metricas_train = avaliar_modelo(y_train, y_train_pred, y_train_pred_proba, "TREINO")
metricas_val = avaliar_modelo(y_val, y_val_pred, y_val_pred_proba, "VALIDAÇÃO")
metricas_test = avaliar_modelo(y_test, y_test_pred, y_test_pred_proba, "TESTE")

## 10. Matriz de Confusão

In [ ]:
# Criar figura com matrizes de confusão
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

conjuntos = [
    (y_train, y_train_pred, "Treino"),
    (y_val, y_val_pred, "Validação"),
    (y_test, y_test_pred, "Teste")
]

for idx, (y_true, y_pred, nome) in enumerate(conjuntos):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f"Matriz de Confusão - {nome}", fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Valor Real')
    axes[idx].set_xlabel('Predição')
    axes[idx].set_xticklabels(['Sem Churn', 'Com Churn'])
    axes[idx].set_yticklabels(['Sem Churn', 'Com Churn'])

plt.tight_layout()
plt.show()

print("\nInterpretação da Matriz de Confusão:")
print("- Verdadeiros Negativos (TN): Clientes sem churn preditos corretamente")
print("- Falsos Positivos (FP): Clientes sem churn preditos como churn")
print("- Falsos Negativos (FN): Clientes com churn preditos como sem churn")
print("- Verdadeiros Positivos (TP): Clientes com churn preditos corretamente")

## 11. Curva ROC

In [ ]:
# Calcular curva ROC para todos os conjuntos
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred_proba)
fpr_val, tpr_val, _ = roc_curve(y_val, y_val_pred_proba)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_proba)

# Plotar curvas ROC
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(fpr_train, tpr_train, label=f'Treino (AUC={metricas_train["ROC-AUC"]:.4f})', linewidth=2)
ax.plot(fpr_val, tpr_val, label=f'Validação (AUC={metricas_val["ROC-AUC"]:.4f})', linewidth=2)
ax.plot(fpr_test, tpr_test, label=f'Teste (AUC={metricas_test["ROC-AUC"]:.4f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório (AUC=0.5000)')

ax.set_xlabel('Taxa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos', fontsize=12)
ax.set_title('Curva ROC - XGBoost para Predição de Churn', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("A curva ROC mostra a relação entre sensibilidade (TPR) e especificidade (1-FPR)")
print("AUC > 0.7: Bom desempenho")
print("AUC > 0.8: Excelente desempenho")

## 12. Importância das Features

In [ ]:
# XGBoost Feature Importance
# Padrão: GAIN
# Redução total de perda (loss) causada pela feature
# Calcula: Ganho médio quando a feature é usada em divisões
# Fórmula: Ganho = Loss_pai - (Loss_esquerda + Loss_direita)
# Interpretação: Features com maior redução de erro são mais importantes

In [ ]:
# Obter importância das features
feature_importance = modelo_xgb.feature_importances_
feature_names = X_train.columns

# Criar dataframe com importância
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importância': feature_importance
}).sort_values('Importância', ascending=False)


# Plotar top 15 features
fig, ax = plt.subplots(figsize=(10, 8))
top_n = 15
importance_top = importance_df.head(top_n)

ax.barh(range(len(importance_top)), importance_top['Importância'].values)
ax.set_yticks(range(len(importance_top)))
ax.set_yticklabels(importance_top['Feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Importância', fontsize=12)
ax.set_title(f'Top {top_n} Features Mais Importantes', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAs {top_n} features mais importantes representam {importance_top['Importância'].sum():.2%} da importância total")

## 13. Relatório de Classificação Detalhado

In [ ]:
# Relatório detalhado para o conjunto de teste
print("RELATÓRIO DE CLASSIFICAÇÃO - CONJUNTO DE TESTE")
print("="*80)
print(classification_report(y_test, y_test_pred, target_names=['Sem Churn', 'Com Churn']))

print("\nInterpretação:")
print("- Precision: Dentre as predições de churn, quantas estavam corretas?")
print("- Recall (Sensibilidade): Dentre os clientes que fizeram churn, quantos foram identificados?")
print("- F1-Score: Média harmônica entre Precision e Recall")
print("- Support: Número de amostras em cada classe")

## 14. Exemplos de Predições

In [ ]:
# Criar dataframe com predições e probabilidades no conjunto de teste
resultados_teste = pd.DataFrame({
    'Churn_Real': y_test.values,
    'Churn_Predito': y_test_pred,
    'Probabilidade_Churn': y_test_pred_proba,
    'Probabilidade_Sem_Churn': 1 - y_test_pred_proba
})

# Adicionar features originais
resultados_teste = pd.concat([X_test_scaled.reset_index(drop=True), resultados_teste], axis=1)

# Mostrar exemplos de diferentes cenários
print("EXEMPLOS DE PREDIÇÕES - CONJUNTO DE TESTE")
print("="*100)

print("\n1. Predições CORRETAS de CHURN (Verdadeiros Positivos):")
corretos_churn = resultados_teste[(resultados_teste['Churn_Real']==1) & (resultados_teste['Churn_Predito']==1)]
print(corretos_churn[['Churn_Real', 'Churn_Predito', 'Probabilidade_Churn']].head(5))

print("\n2. Predições CORRETAS de SEM CHURN (Verdadeiros Negativos):")
corretos_sem_churn = resultados_teste[(resultados_teste['Churn_Real']==0) & (resultados_teste['Churn_Predito']==0)]
print(corretos_sem_churn[['Churn_Real', 'Churn_Predito', 'Probabilidade_Churn']].head(5))

print("\n3. Predições INCORRETAS (Erros do Modelo):")
erros = resultados_teste[resultados_teste['Churn_Real'] != resultados_teste['Churn_Predito']]
print(erros[['Churn_Real', 'Churn_Predito', 'Probabilidade_Churn']].head(10))
print(f"\nTotal de erros: {len(erros)} de {len(resultados_teste)} ({len(erros)/len(resultados_teste)*100:.2f}%)")

## 15. Comparação de Desempenho entre Conjuntos

In [ ]:
# Criar dataframe com comparação de métricas
comparacao_df = pd.DataFrame({
    'Treino': metricas_train,
    'Validação': metricas_val,
    'Teste': metricas_test
})

print("COMPARAÇÃO DE DESEMPENHO ENTRE CONJUNTOS")
print("="*80)
print(comparacao_df.round(4))

# Visualizar comparação
fig, ax = plt.subplots(figsize=(12, 6))
comparacao_df.T.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Comparação de Métricas entre Conjuntos', fontsize=14, fontweight='bold')
ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Conjunto', fontsize=12)
ax.legend(title='Métrica', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nInterpretação:")
print("- Se Treino >> Teste: Possível overfitting")
print("- Se Treino ≈ Validação ≈ Teste: Modelo generaliza bem")
print("- Se todos são baixos: Underfitting")